# 🧪 Gas Sensor Drift Compensation using Machine Learning

---

**Dataset:** UCI Gas Sensor Array Drift Dataset (Real Data)  
**Domain:** Machine Learning · Sensor Data · Concept Drift  
**Libraries:** NumPy, Pandas, Matplotlib, Scikit-learn  

---

## 📖 Project Overview

This project uses the real-world **UCI Gas Sensor Array Drift Dataset** — 13,910 readings collected over 36 months across 10 measurement batches from a 16-sensor array. Due to physical degradation of sensor materials, sensor responses drift progressively over time.

The challenge: a model trained on early batches must classify gases in later batches where sensor characteristics have shifted.

### Gas Classes
| Code | Gas | Code | Gas |
|------|-------------|------|-------------|
| 1 | Ethanol | 4 | Acetone |
| 2 | Ethylene | 5 | Acetaldehyde |
| 3 | Ammonia | 6 | Toluene |

> ⚠️ Some gases are absent from certain batches — this reflects real collection constraints in the original experiment.

### Key ML Concepts
| Concept | Definition |
|--------|------------|
| **Covariate Shift** | P(X) changes over time; P(Y\|X) remains stable |
| **Concept Drift** | The relationship P(Y\|X) itself changes |
| **Temporal Leakage** | Using future data in training — inflates accuracy dishonestly |
| **Distribution Mismatch** | Train and test come from different distributions |

### Project Story
```
Load Real Data → EDA → Time-based Split → Baseline Fails
    → Visualise Drift (PCA) → Apply 2 Compensation Methods
        → Compare All Results → Conclude
```

---
## 📦 Section 0 — Library Imports

In [ ]:
# ============================================================
# SECTION 0: IMPORTS
# ============================================================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': 'white'
})

# Gas label mapping (from UCI dataset documentation)
GAS_NAMES = {1:'Ethanol', 2:'Ethylene', 3:'Ammonia',
             4:'Acetone', 5:'Acetaldehyde', 6:'Toluene'}
PALETTE   = ['#2196F3','#4CAF50','#FF5722','#9C27B0','#FF9800','#00BCD4']

print('✅ Libraries imported.')
print(f'   NumPy {np.__version__} | Pandas {pd.__version__}')

---
## 📂 Section 1 — Data Loading

The dataset uses **LibSVM sparse format**. Each row represents one sensor measurement:
```
<gas_label>  1:<value>  2:<value>  ...  128:<value>
```
- **Label (1–6):** the gas being measured
- **Features 1–128:** 8 statistical descriptors (DR, NI, DN, LN, SD, M, AM, LNSD) from each of 16 sensors

We load all 10 batch files and concatenate them, adding a `batch` column to preserve temporal order.

In [ ]:
# ============================================================
# SECTION 1: LOAD BATCH FILES
# ============================================================

DATA_DIR    = 'Dataset'      # Folder with batch1.dat ... batch10.dat
N_BATCHES   = 10
N_FEATURES  = 128
FEAT_COLS   = [f'f{i}' for i in range(1, N_FEATURES + 1)]

def parse_batch(filepath, batch_id):
    """Parse one LibSVM .dat file into a DataFrame."""
    records = []
    with open(filepath) as fh:
        for line in fh:
            parts = line.strip().split()
            if not parts:
                continue
            label = int(parts[0])
            feat  = {int(k): float(v)
                     for k, v in (p.split(':') for p in parts[1:])}
            row   = [feat.get(i, np.nan) for i in range(1, N_FEATURES+1)]
            records.append([label] + row + [batch_id])
    return pd.DataFrame(records, columns=['label']+FEAT_COLS+['batch'])


print('🔄 Loading batch files...')
dfs = []
for b in range(1, N_BATCHES+1):
    df_b = parse_batch(os.path.join(DATA_DIR, f'batch{b}.dat'), b)
    dfs.append(df_b)
    labs = sorted(df_b['label'].unique())
    print(f'   batch{b:>2}.dat  →  {len(df_b):>5} samples | '
          f'gases: {[GAS_NAMES[l] for l in labs]}')

df = pd.concat(dfs, ignore_index=True)
df['gas_name'] = df['label'].map(GAS_NAMES)

print(f'\n✅ Full dataset: {df.shape[0]:,} samples × {df.shape[1]} columns')
df.head(3)

In [ ]:
# ── Prepare arrays ────────────────────────────────────────────
X_all     = df[FEAT_COLS].values          # (N, 128)
y_raw     = df['label'].values            # integer labels 1–6
batch_all = df['batch'].values            # batch IDs 1–10

# Encode labels to 0-based indices for sklearn
le      = LabelEncoder()
y_enc   = le.fit_transform(y_raw)
CLASS_NAMES = [GAS_NAMES[c] for c in le.classes_]
ALL_BATCHES = sorted(df['batch'].unique())

print('✅ Arrays ready:')
print(f'   X shape   : {X_all.shape}')
print(f'   Classes   : {list(zip(le.classes_, CLASS_NAMES))}')

---
## 📊 Section 2 — Exploratory Data Analysis

Understanding the data before modelling is essential. We need to know:
- Are classes balanced across batches?
- How large is each batch?
- Are there missing values?
- Which gases appear in which batches?

In [ ]:
# ── 2.1 Summary Statistics ────────────────────────────────────
print('='*60)
print('  DATASET SUMMARY')
print('='*60)
print(f'  Total samples  : {len(df):,}')
print(f'  Features       : {len(FEAT_COLS)}')
print(f'  Gas classes    : {df["label"].nunique()}')
print(f'  Batches        : {df["batch"].nunique()}')
print(f'  Missing values : {df[FEAT_COLS].isnull().sum().sum()}')
print()
print('  Class distribution:')
for code, name in GAS_NAMES.items():
    n   = int((y_raw == code).sum())
    pct = n / len(df) * 100
    bar = '█' * int(pct/1.5)
    print(f'    {code} {name:<14}: {n:>5}  ({pct:5.1f}%)  {bar}')
print()
print('  Samples per batch:')
for b in ALL_BATCHES:
    print(f'    Batch {b:>2}: {(batch_all==b).sum():>5}')

In [ ]:
# ── 2.2 EDA Plots ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('EDA — UCI Gas Sensor Array Drift Dataset',
             fontsize=14, fontweight='bold', y=1.01)

# Plot 1: Overall class distribution
ax = axes[0]
counts = [(y_raw == c).sum() for c in le.classes_]
bars = ax.bar(CLASS_NAMES, counts, color=PALETTE, edgecolor='white', lw=1.2)
ax.set_title('Overall Class Distribution')
ax.set_xlabel('Gas')
ax.set_ylabel('Samples')
ax.tick_params(axis='x', rotation=35)
for bar, val in zip(bars, counts):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+15,
            str(val), ha='center', va='bottom', fontsize=8.5)
ax.set_ylim(0, max(counts)*1.18)

# Plot 2: Samples per batch
ax = axes[1]
b_counts = [(batch_all==b).sum() for b in ALL_BATCHES]
cols_b   = plt.cm.viridis(np.linspace(0.15, 0.9, len(ALL_BATCHES)))
ax.bar(ALL_BATCHES, b_counts, color=cols_b, edgecolor='white')
ax.set_title('Samples per Batch (Time →)')
ax.set_xlabel('Batch Number')
ax.set_ylabel('Samples')
ax.set_xticks(ALL_BATCHES)
for x, yv in zip(ALL_BATCHES, b_counts):
    ax.text(x, yv+15, str(yv), ha='center', va='bottom', fontsize=8)
ax.set_ylim(0, max(b_counts)*1.15)

# Plot 3: Class x Batch heatmap
ax = axes[2]
pivot = df.groupby(['batch','gas_name']).size().unstack(fill_value=0)
ordered = [GAS_NAMES[c] for c in sorted(GAS_NAMES) if GAS_NAMES[c] in pivot.columns]
pivot = pivot[ordered]
im = ax.imshow(pivot.T.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(pivot.index)))
ax.set_xticklabels([f'B{b}' for b in pivot.index])
ax.set_yticks(range(len(pivot.columns)))
ax.set_yticklabels(pivot.columns)
ax.set_title('Sample Count Heatmap\n(Class × Batch)')
ax.set_xlabel('Batch (Time →)')
plt.colorbar(im, ax=ax, shrink=0.8, label='Count')
for i, col in enumerate(pivot.columns):
    for j, b in enumerate(pivot.index):
        val = pivot.loc[b, col]
        ax.text(j, i, str(val), ha='center', va='center', fontsize=7,
                color='white' if val > 500 else 'black')

plt.tight_layout()
plt.savefig('eda_plots.png', bbox_inches='tight')
plt.show()

print('📌 Observations:')
print('   1. Dataset is IMBALANCED — Ethanol/Ethylene dominate later batches.')
print('   2. Some gases absent from batches 3–5 (real collection constraint).')
print('   3. Batch sizes vary greatly (161–3,613) — real-world irregularity.')
print('   4. This is the ACTUAL UCI dataset, not synthetic data.')

---
## ⚙️ Section 3 — Preprocessing

### Missing Values
The UCI dataset is complete — all 128 features are present in every row.

### Feature Scaling
The 128 features span vastly different scales:
- `f1` (raw resistance DR): values in the thousands (e.g. 15,000–58,000)
- `f2` (normalised index NI): values around 1–5

**StandardScaler** brings all features to mean=0, std=1:
$$x' = \frac{x - \mu_{train}}{\sigma_{train}}$$

⚠️ **Critical:** The scaler is **fit only on training data** to avoid data leakage.

In [ ]:
# ── 3.1 Missing value check ───────────────────────────────────
missing = df[FEAT_COLS].isnull().sum().sum()
print(f'Missing values: {missing}  →  No imputation needed.')

# ── 3.2 Feature range before scaling ─────────────────────────
print('\nFeature value ranges (selected features):')
print(df[['f1','f2','f17','f65','f128']].agg(['min','max','mean','std']).round(2).to_string())
print('\n→ f1 is in the tens of thousands; f2 is ~1–5.')
print('  Scaling is essential to prevent high-magnitude features from dominating.')

---
## ✂️ Section 4 — Time-Based Train / Test Split

### Why NOT a random split?

```python
# ❌ WRONG for temporal data
train_test_split(X, y, test_size=0.3, shuffle=True)
```
A random split mixes all batches into both sets, letting the model "see" future drift patterns during training. This produces inflated accuracy that **does not reflect real deployment**.

### Chronological split
We train on the **first 7 batches** (earlier time periods) and test on the **last 3 batches** (later time periods).

```
Timeline ──────────────────────────────────────────────────►
  Batch: [1][2][3][4][5][6][7]   |   [8][9][10]
          ◄─── TRAINING ────────►   ◄─── TEST ───►
                                 ↑
                          Deployment point
```

In [ ]:
# ============================================================
# SECTION 4: TIME-BASED SPLIT
# ============================================================
TRAIN_BATCHES = [1, 2, 3, 4, 5, 6, 7]
TEST_BATCHES  = [8, 9, 10]

tr_mask = np.isin(batch_all, TRAIN_BATCHES)
te_mask = np.isin(batch_all, TEST_BATCHES)

X_train = X_all[tr_mask];  y_train = y_enc[tr_mask];  b_train = batch_all[tr_mask]
X_test  = X_all[te_mask];  y_test  = y_enc[te_mask];  b_test  = batch_all[te_mask]

# Fit scaler ONLY on training data
g_scaler    = StandardScaler()
X_train_sc  = g_scaler.fit_transform(X_train)
X_test_sc   = g_scaler.transform(X_test)

print('✅ Time-based split:')
print(f'   TRAIN  Batches {TRAIN_BATCHES}  →  {len(X_train):,} samples  ({len(X_train)/len(X_all)*100:.1f}%)')
print(f'   TEST   Batches {TEST_BATCHES}   →  {len(X_test):,} samples  ({len(X_test)/len(X_all)*100:.1f}%)')
print(f'\n   Classes in train set : {sorted(set(y_train))}')
print(f'   Classes in test set  : {sorted(set(y_test))}')
print(f'\n   ✅ Scaler fitted only on training data — no leakage.')

---
## 🤖 Section 5 — Baseline Model

A **Random Forest** trained on all training batches with global (non-batch-aware) scaling. No drift compensation — this is our performance floor.

**Why Random Forest?**
- Handles 128 correlated features well
- Strong ensemble baseline; hard to beat trivially
- Provides feature importance for interpretability

In [ ]:
# ============================================================
# SECTION 5: BASELINE MODEL
# ============================================================
print('🏋️  Training Baseline Random Forest...')

bl_rf = RandomForestClassifier(
    n_estimators=200, max_depth=20,
    min_samples_leaf=2,
    random_state=RANDOM_STATE, n_jobs=-1
)
bl_rf.fit(X_train_sc, y_train)

y_pred_bl   = bl_rf.predict(X_test_sc)
baseline_acc = accuracy_score(y_test, y_pred_bl)

print(f'\n{"="*55}')
print(f'  BASELINE  |  Test Batches: {TEST_BATCHES}')
print(f'  Test Accuracy : {baseline_acc*100:.2f}%')
print(f'{"="*55}')
print()
print(classification_report(y_test, y_pred_bl,
                             target_names=CLASS_NAMES, zero_division=0))

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8,6))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred_bl),
    display_labels=CLASS_NAMES
).plot(ax=ax, colorbar=True, cmap='Blues', values_format='d')
ax.set_title(f'Baseline Confusion Matrix  |  Accuracy: {baseline_acc*100:.2f}%',
             fontweight='bold')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig('confusion_baseline.png', bbox_inches='tight')
plt.show()

---
## 📉 Section 6 — Drift Demonstration

We evaluate the baseline model **batch by batch** to confirm that performance degrades over time. A downward trend in the test period is direct evidence that **sensor drift is causing distribution mismatch**.

> If this graph shows no trend, drift is not a problem. If it shows a clear drop — as expected — drift compensation is justified.

In [ ]:
# ── Per-batch accuracy ────────────────────────────────────────
print('📊 Baseline per-batch accuracy (using global scaler):\n')
pb_acc_bl = {}
for b in ALL_BATCHES:
    m   = batch_all == b
    Xb  = g_scaler.transform(X_all[m])
    acc = accuracy_score(y_enc[m], bl_rf.predict(Xb))
    pb_acc_bl[b] = acc
    tag = '← TRAIN' if b in TRAIN_BATCHES else '← TEST ⚠️'
    print(f'   Batch {b:>2} : {acc*100:6.2f}%  {tag}')

avg_tr = np.mean([pb_acc_bl[b] for b in TRAIN_BATCHES])
avg_te = np.mean([pb_acc_bl[b] for b in TEST_BATCHES])
print(f'\n   Mean TRAIN accuracy : {avg_tr*100:.2f}%')
print(f'   Mean TEST  accuracy : {avg_te*100:.2f}%')
print(f'   Drift-induced drop  : {(avg_tr-avg_te)*100:.2f} pp')

In [ ]:
# ── Accuracy vs Batch Plot ────────────────────────────────────
fig, ax = plt.subplots(figsize=(11,5))

accs = [pb_acc_bl[b]*100 for b in ALL_BATCHES]

ax.axvspan(min(TRAIN_BATCHES)-.5, max(TRAIN_BATCHES)+.5, alpha=.08,
           color='#2196F3', label='Training Period')
ax.axvspan(min(TEST_BATCHES)-.5,  max(TEST_BATCHES)+.5,  alpha=.08,
           color='#FF5722', label='Test Period (Deployment)')

ax.plot(ALL_BATCHES, accs, '-o', color='#333', lw=2, zorder=3)
for b in TRAIN_BATCHES:
    ax.scatter(b, pb_acc_bl[b]*100, color='#2196F3', s=90, zorder=4)
for b in TEST_BATCHES:
    ax.scatter(b, pb_acc_bl[b]*100, color='#FF5722', s=90, zorder=4)

ax.axhline(avg_tr*100, xmin=0,    xmax=.73, color='#2196F3', ls='--', alpha=.5, lw=1.5)
ax.axhline(avg_te*100, xmin=.74, xmax=1,   color='#FF5722', ls='--', alpha=.5, lw=1.5)

ax.annotate(f'Drift drop ≈ {(avg_tr-avg_te)*100:.1f} pp',
            xy=(TEST_BATCHES[0], pb_acc_bl[TEST_BATCHES[0]]*100),
            xytext=(TEST_BATCHES[0]-1, pb_acc_bl[TEST_BATCHES[0]]*100+12),
            arrowprops=dict(arrowstyle='->', color='#FF5722'),
            color='#FF5722', fontsize=10)

ax.set_title('Baseline: Accuracy vs Batch — Evidence of Sensor Drift',
             fontweight='bold')
ax.set_xlabel('Batch Number (Time →)')
ax.set_ylabel('Accuracy (%)')
ax.set_xticks(ALL_BATCHES)
ax.set_xticklabels([f'Batch {b}' for b in ALL_BATCHES], rotation=30)
ax.set_ylim(0, 120)
ax.legend(loc='lower left')
ax.grid(True, alpha=.25)

plt.tight_layout()
plt.savefig('accuracy_vs_batch.png', bbox_inches='tight')
plt.show()

---
## 🔭 Section 7 — Drift Visualisation with PCA

PCA reduces 128 features to 2 dimensions while preserving maximum variance. If drift is present, samples from different batches will form **separate, shifting clusters** in this 2D space — even though PCA has no knowledge of batch labels.

**This is the most visually compelling evidence of covariate shift.**

In [ ]:
# ── Fit PCA ───────────────────────────────────────────────────
X_all_sc = g_scaler.transform(X_all)
pca      = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca    = pca.fit_transform(X_all_sc)
ev       = pca.explained_variance_ratio_

print(f'PCA: PC1={ev[0]*100:.2f}%  PC2={ev[1]*100:.2f}%  Total={sum(ev)*100:.2f}% variance')

In [ ]:
# ── PCA Scatter: batch vs class ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('PCA Drift Visualisation — UCI Real Data',
             fontsize=14, fontweight='bold')

idx = np.random.choice(len(X_pca), size=min(5000, len(X_pca)), replace=False)

# Left: colour by batch
ax = axes[0]
sc = ax.scatter(X_pca[idx,0], X_pca[idx,1],
                c=batch_all[idx], cmap='RdYlGn_r', alpha=.45, s=7)
plt.colorbar(sc, ax=ax, label='Batch Number')
ax.set_title('Coloured by BATCH\n(Batch clusters reveal drift)',
             fontweight='bold', fontsize=11)
ax.set_xlabel(f'PC1 ({ev[0]*100:.1f}% var.)')
ax.set_ylabel(f'PC2 ({ev[1]*100:.1f}% var.)')
ax.text(.02,.97,'Green=Early  Red=Late', transform=ax.transAxes,
        fontsize=9, va='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=.7))

# Right: colour by class
ax = axes[1]
for i, (cls, col) in enumerate(zip(CLASS_NAMES, PALETTE)):
    m = y_enc[idx] == i
    ax.scatter(X_pca[idx[m],0], X_pca[idx[m],1],
               c=col, alpha=.35, s=7, label=cls)
ax.set_title('Coloured by GAS CLASS\n(Classes separable despite drift)',
             fontweight='bold', fontsize=11)
ax.set_xlabel(f'PC1 ({ev[0]*100:.1f}% var.)')
ax.set_ylabel(f'PC2 ({ev[1]*100:.1f}% var.)')
ax.legend(markerscale=2.5, fontsize=9)

plt.tight_layout()
plt.savefig('pca_viz.png', bbox_inches='tight')
plt.show()

print('📌 Left:  Batches form non-overlapping clusters → confirmed covariate shift.')
print('   Right: Gas classes still separable within batches → P(Y|X) stable.')
print('   Diagnosis: COVARIATE SHIFT, not concept drift. Compensation can fix this.')

In [ ]:
# ── Batch centroid trajectory ─────────────────────────────────
fig, ax = plt.subplots(figsize=(8,6))

centroids = np.array([X_pca[batch_all==b].mean(0) for b in ALL_BATCHES])
ax.plot(centroids[:,0], centroids[:,1], '-', color='#bbb', lw=1.5, zorder=2)

traj_c = plt.cm.RdYlGn_r(np.linspace(.1, .9, len(ALL_BATCHES)))
for i, (b, c) in enumerate(zip(ALL_BATCHES, traj_c)):
    ax.scatter(*centroids[i], color=c, s=130, zorder=3,
               edgecolors='white', lw=1.2)
    ax.annotate(f'B{b}', xy=centroids[i],
                xytext=(centroids[i,0]+.05, centroids[i,1]+.05),
                fontsize=9.5, fontweight='bold')

mid_x = (centroids[6,0]+centroids[7,0])/2
ax.axvline(mid_x, color='red', ls='--', alpha=.4, label='Train/Test split')

ax.set_title('Batch Centroid Trajectory in PCA Space\n'
             '(Each point = centre of all samples in that batch)',
             fontweight='bold')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(); ax.grid(True, alpha=.2)
plt.tight_layout()
plt.savefig('pca_trajectory.png', bbox_inches='tight')
plt.show()

print('📌 The trajectory shows a PROGRESSIVE, SYSTEMATIC shift — classic sensor drift.')

---
## 🔧 Section 8 — Drift Compensation Methods

### Method A: Batch-wise Normalisation

**Core idea:** Normalise each batch independently, removing batch-specific mean and variance.

$$x'_{i,b} = \frac{x_{i,b} - \mu_b}{\sigma_b}$$

**Why it works:** Drift primarily shifts the mean operating point of each sensor. Batch-wise normalisation removes this offset, aligning the feature distributions across time.

**Practical note:** At test time, $\mu_b$ and $\sigma_b$ are computed from the test batch itself — realistic in practice, as a small unlabelled calibration set is often available.

In [ ]:
# ============================================================
# METHOD A: BATCH-WISE NORMALISATION
# ============================================================
print('⚙️  Method A: Batch-wise Normalisation')
print('─'*52)

def batch_normalize(X, batches, batch_ids):
    """Normalise features within each batch independently."""
    X_out = X.copy().astype(float)
    for b in batch_ids:
        m = batches == b
        X_out[m] = StandardScaler().fit_transform(X[m])
    return X_out

X_tr_bwn = batch_normalize(X_train, b_train, TRAIN_BATCHES)
X_te_bwn = batch_normalize(X_test,  b_test,  TEST_BATCHES)

rf_bwn = RandomForestClassifier(
    n_estimators=200, max_depth=20,
    min_samples_leaf=2,
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_bwn.fit(X_tr_bwn, y_train)

y_pred_bwn = rf_bwn.predict(X_te_bwn)
acc_bwn    = accuracy_score(y_test, y_pred_bwn)

print(f'   Test Accuracy : {acc_bwn*100:.2f}%')
print(f'   vs Baseline   : {(acc_bwn-baseline_acc)*100:+.2f} pp')

In [ ]:
# ── Method A per-batch ────────────────────────────────────────
pb_acc_bwn = {}
print('   Per-batch accuracy (Method A):')
for b in ALL_BATCHES:
    m   = batch_all == b
    Xb  = StandardScaler().fit_transform(X_all[m])
    acc = accuracy_score(y_enc[m], rf_bwn.predict(Xb))
    pb_acc_bwn[b] = acc
    tag = 'TRAIN' if b in TRAIN_BATCHES else 'TEST '
    print(f'      Batch {b:>2} ({tag}) → {acc*100:.2f}%')

---
### Method B: Sliding Window Training

**Core idea:** Train only on the **most recent training batches** (those closest in time to the test period).

```
Full training:    [B1][B2][B3][B4][B5][B6][B7]  → TEST [B8][B9][B10]
Sliding window:               [B5][B6][B7]       → TEST [B8][B9][B10]
                               ↑ window = last 3 batches
```

**Why it works:** Recent batches have distributions closest to upcoming test data. Old data can actively hurt the model by anchoring it to outdated patterns.

**Trade-off:** Less training data — can hurt if the window contains rare classes.

In [ ]:
# ============================================================
# METHOD B: SLIDING WINDOW TRAINING
# ============================================================
print('⚙️  Method B: Sliding Window Training')
print('─'*52)

WINDOW      = 3
WIN_BATCHES = TRAIN_BATCHES[-WINDOW:]   # [5, 6, 7]
print(f'   Window: last {WINDOW} training batches = {WIN_BATCHES}')

win_m      = np.isin(batch_all, WIN_BATCHES)
X_tr_win   = X_all[win_m]
y_tr_win   = y_enc[win_m]
print(f'   Window samples: {len(X_tr_win):,}  (vs {len(X_train):,} full)')

sc_win     = StandardScaler()
X_tr_win_s = sc_win.fit_transform(X_tr_win)
X_te_win_s = sc_win.transform(X_test)

rf_win = RandomForestClassifier(
    n_estimators=200, max_depth=20,
    min_samples_leaf=2,
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_win.fit(X_tr_win_s, y_tr_win)

y_pred_win = rf_win.predict(X_te_win_s)
acc_win    = accuracy_score(y_test, y_pred_win)

print(f'   Test Accuracy : {acc_win*100:.2f}%')
print(f'   vs Baseline   : {(acc_win-baseline_acc)*100:+.2f} pp')

In [ ]:
# ── Method B per-batch ────────────────────────────────────────
pb_acc_win = {}
print('   Per-batch accuracy (Method B):')
for b in ALL_BATCHES:
    m   = batch_all == b
    Xb  = sc_win.transform(X_all[m])
    acc = accuracy_score(y_enc[m], rf_win.predict(Xb))
    pb_acc_win[b] = acc
    tag = 'TRAIN' if b in TRAIN_BATCHES else 'TEST '
    print(f'      Batch {b:>2} ({tag}) → {acc*100:.2f}%')

---
## 📊 Section 9 — Comparison of All Methods

In [ ]:
# ── Summary Table ─────────────────────────────────────────────
all_results = [
    ('Baseline (Global Scaling)',              baseline_acc, pb_acc_bl),
    ('Method A: Batch-wise Normalisation',     acc_bwn,      pb_acc_bwn),
    ('Method B: Sliding Window (last 3)',      acc_win,      pb_acc_win),
]

print('='*68)
print(f"  {'Method':<40} {'Test Acc':>10} {'vs Baseline':>12}")
print('='*68)
for name, acc, _ in all_results:
    delta = acc - baseline_acc
    sign  = '+' if delta >= 0 else ''
    print(f"  {name:<40} {acc*100:>9.2f}%  {sign}{delta*100:>9.2f}pp")
print('='*68)

best_acc    = max(acc_bwn, acc_win)
best_method = 'Method A' if acc_bwn >= acc_win else 'Method B'
print(f'\n  🏆 Best: {best_method}  →  {best_acc*100:.2f}%')
print(f'     Gain over baseline: {(best_acc-baseline_acc)*100:+.2f} pp')

# DataFrame for display
pd.DataFrame({
    'Method':          [r[0] for r in all_results],
    'Test Acc (%)':    [round(r[1]*100,2) for r in all_results],
    'Improvement (pp)':[round((r[1]-baseline_acc)*100,2) for r in all_results],
    'Strategy':        ['No compensation',
                        'Remove per-batch mean/variance shift',
                        f'Train on last {WINDOW} batches only']
})

In [ ]:
# ── Visual Comparison ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16,6))
fig.suptitle('Drift Compensation — Method Comparison (UCI Real Data)',
             fontsize=14, fontweight='bold')

# Left: bar chart
ax = axes[0]
labels = ['Baseline','Method A\n(Batch Norm)','Method B\n(Sliding Win)']
accs_  = [baseline_acc*100, acc_bwn*100, acc_win*100]
bcols  = ['#e74c3c','#3498db','#27ae60']
bars   = ax.bar(labels, accs_, color=bcols, edgecolor='white', lw=1.5, width=.5)
ax.axhline(baseline_acc*100, color='#e74c3c', ls='--', alpha=.4, lw=1.5)
for bar, v in zip(bars, accs_):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+.4,
            f'{v:.2f}%', ha='center', va='bottom', fontweight='bold')
ax.set_title('Overall Test Set Accuracy')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, max(accs_)*1.2)
ax.grid(True, axis='y', alpha=.3)

# Right: per-batch lines
ax = axes[1]
ax.plot(ALL_BATCHES, [pb_acc_bl[b]*100  for b in ALL_BATCHES],
        '-o', color='#e74c3c', lw=2, ms=7, label='Baseline')
ax.plot(ALL_BATCHES, [pb_acc_bwn[b]*100 for b in ALL_BATCHES],
        '-s', color='#3498db', lw=2, ms=7, label='Method A')
ax.plot(ALL_BATCHES, [pb_acc_win[b]*100 for b in ALL_BATCHES],
        '-^', color='#27ae60', lw=2, ms=7, label='Method B')
ax.axvspan(min(TRAIN_BATCHES)-.4, max(TRAIN_BATCHES)+.4, alpha=.06, color='#2196F3')
ax.axvspan(min(TEST_BATCHES)-.4,  max(TEST_BATCHES)+.4,  alpha=.06, color='#FF5722')
ax.text(np.mean(TRAIN_BATCHES), 109, 'TRAIN', ha='center',
        color='#2196F3', fontweight='bold')
ax.text(np.mean(TEST_BATCHES),  109, 'TEST',  ha='center',
        color='#FF5722', fontweight='bold')
ax.set_title('Per-Batch Accuracy: All Methods')
ax.set_xlabel('Batch Number (Time →)')
ax.set_ylabel('Accuracy (%)')
ax.set_xticks(ALL_BATCHES)
ax.set_xticklabels([f'B{b}' for b in ALL_BATCHES], rotation=30)
ax.set_ylim(0, 120)
ax.legend(fontsize=9)
ax.grid(True, alpha=.25)

plt.tight_layout()
plt.savefig('method_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Confusion Matrix Trio ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18,5))
fig.suptitle('Confusion Matrices — Test Set (Batches 8, 9, 10)',
             fontsize=13, fontweight='bold')
for (y_pred, title, ax) in [
    (y_pred_bl,  f'Baseline ({baseline_acc*100:.1f}%)',  axes[0]),
    (y_pred_bwn, f'Method A ({acc_bwn*100:.1f}%)',       axes[1]),
    (y_pred_win, f'Method B ({acc_win*100:.1f}%)',        axes[2]),
]:
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_pred),
        display_labels=CLASS_NAMES
    ).plot(ax=ax, colorbar=False, cmap='Blues', values_format='d')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=40)
plt.tight_layout()
plt.savefig('confusion_comparison.png', bbox_inches='tight')
plt.show()

---
## 🎓 Section 10 — Final Conclusion

In [ ]:
# ── Final Summary ─────────────────────────────────────────────
print('\n' + '═'*66)
print('   FINAL RESULTS — UCI GAS SENSOR ARRAY DRIFT DATASET')
print('═'*66)
print(f"  {'Method':<40} {'Test Acc':>9} {'Δ':>8}")
print('─'*66)
icons = ['❌','✅','✅']
for (name, acc, _), icon in zip(all_results, icons):
    d = acc - baseline_acc
    print(f"  {icon} {name:<38} {acc*100:>8.2f}%  {d*100:>+7.2f}pp")
print('═'*66)
print(f'  Dataset   : Real UCI Gas Sensor Array Drift Dataset')
print(f'  Train     : Batches {TRAIN_BATCHES}  ({len(X_train):,} samples)')
print(f'  Test      : Batches {TEST_BATCHES}  ({len(X_test):,} samples)')
print(f'  Split     : Chronological — NO random shuffle')
print('═'*66)

print("""
┌──────────────────────────────────────────────────────────────┐
│  VIVA Q&A                                                    │
├──────────────────────────────────────────────────────────────┤
│  Q: Why is random train-test split wrong here?              │
│  A: It creates temporal leakage — future data enters         │
│     training, inflating accuracy. Real deployment uses       │
│     only past data to predict future sensor readings.        │
│                                                              │
│  Q: Is this covariate shift or concept drift?               │
│  A: Covariate shift. P(X) drifts with sensor degradation,   │
│     but P(Y|X) — the gas-to-sensor relationship — is        │
│     physically stable. PCA shows classes stay separable.     │
│                                                              │
│  Q: Why does batch normalisation help?                       │
│  A: Drift shifts sensor baselines (mean). Per-batch          │
│     standardisation removes this offset, aligning            │
│     distributions across batches.                            │
│                                                              │
│  Q: Why does sliding window help?                            │
│  A: Recent batches are most similar to test data. Old        │
│     batches anchor the model to outdated patterns.           │
│                                                              │
│  Q: What are advanced alternatives?                          │
│  A: CORAL (covariance alignment), DANN (adversarial),        │
│     importance reweighting, online learning with ADWIN.      │
└──────────────────────────────────────────────────────────────┘
""")

In [ ]:
# ── The Full Story Plot ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(13,6))

ax.plot(ALL_BATCHES, [pb_acc_bl[b]*100  for b in ALL_BATCHES],
        '-o', color='#e74c3c', lw=2.5, ms=8, label='Baseline — fails on drifted data')
ax.plot(ALL_BATCHES, [pb_acc_bwn[b]*100 for b in ALL_BATCHES],
        '-s', color='#3498db', lw=2.5, ms=8, label='Method A: Batch-wise Normalisation')
ax.plot(ALL_BATCHES, [pb_acc_win[b]*100 for b in ALL_BATCHES],
        '-^', color='#27ae60', lw=2.5, ms=8, label='Method B: Sliding Window Training')

ax.axvspan(min(TRAIN_BATCHES)-.4, max(TRAIN_BATCHES)+.4, alpha=.07, color='#2196F3')
ax.axvspan(min(TEST_BATCHES)-.4,  max(TEST_BATCHES)+.4,  alpha=.07, color='#FF5722')
ax.text(np.mean(TRAIN_BATCHES), 113, 'TRAINING', ha='center',
        color='#2196F3', fontsize=12, fontweight='bold')
ax.text(np.mean(TEST_BATCHES),  113, 'DEPLOYMENT', ha='center',
        color='#FF5722', fontsize=12, fontweight='bold')

ax.set_title('The Complete Story: Model Fails → Drift Identified → Compensated\n'
             'UCI Gas Sensor Array Drift Dataset (Real Data)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Batch Number (Time →)', fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_xticks(ALL_BATCHES)
ax.set_xticklabels([f'Batch {b}' for b in ALL_BATCHES], rotation=30)
ax.set_ylim(0, 125)
ax.legend(loc='lower left', fontsize=10)
ax.grid(True, alpha=.25)
plt.tight_layout()
plt.savefig('final_summary.png', bbox_inches='tight')
plt.show()

print('✅ Notebook complete. All plots saved.')
print('   Using REAL UCI Gas Sensor Array Drift Dataset.')

---
## 📚 References

1. **Dataset:** Vergara, A., Vembu, S., Ayhan, T., Ryan, M. A., Homer, M. L., & Huerta, R. (2012). *Chemical gas sensor drift compensation using classifier ensembles.* Sensors and Actuators B: Chemical, 166, 320–329.

2. **UCI Repository:** https://archive.ics.uci.edu/ml/datasets/gas+sensor+array+drift+dataset

3. **Concept Drift Survey:** Gama, J., et al. (2014). *A survey on concept drift adaptation.* ACM Computing Surveys.

4. **Covariate Shift:** Shimodaira, H. (2000). *Improving predictive inference under covariate shift.* Journal of Statistical Planning and Inference.

---
*End of Notebook — All results produced from the real UCI Gas Sensor Array Drift Dataset*